<a href="https://colab.research.google.com/github/RavangDai/Farshore/blob/main/colab/farshore_model_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FARSHORE model server on Google Colab

Run every AI job in FARSHORE on a free Colab GPU when your laptop is too slow.

| Job in the game | Model (T4 preset) | Why this one |
|---|---|---|
| **Planner**: each rival's plan, Antinous' shots, the Sirens' melodies, Polyphemus' sections, Athena, the debriefs | `qwen3.5:9b` | The strongest instruction follower that fits a T4 beside the others: IFEval 91.5 and MMLU-Pro 82.5 on Qwen's card, against about 60 MMLU-Pro for `gemma3:12b`. |
| **Commentary**: the live lines | `qwen3.5:4b` | Small and fast (IFEval 89.8). A line has to arrive while the moment is still on screen. |
| **The Oracle's Key**: writes a multiple-choice question from two retrieved passages of the *Odyssey* and quotes its evidence word for word | `qwen3.5:9b` (the planner; no extra memory) | The game checks every quote and answer against the passage and throws out a question that fails, so precise instruction following matters more than speed. |
| **The Oracle's passage search** (retrieval embeddings) | `embeddinggemma:300m` | On FARSHORE's own retrieval test (`scripts/rag-eval.mjs`, 25 questions), with keywords it ranks the answering passage first 68% of the time. Keywords alone manage 60%, and the old in-game MiniLM hybrid 44%. |
| **Voice** | Kokoro-82M | Still the most used open TTS model, and the game's voices (`bm_george` and the others) are Kokoro's. |

The vendor scores above are measured with thinking on. The game turns thinking off for speed, so check your choice on the game's own requests with `scripts/ai-eval.mjs` (see the notes at the end). The **Proven** preset is the setup measured on 2026-11-10: `gemma3:12b` gave 100% valid replies with a median of 4.0 s on a T4.

The notebook starts:

1. **Ollama**, serving the chat models and the embedding model.
2. **Kokoro** for the voices.
3. A **gateway** that requires an API key and serves the game's four endpoints: chat, embeddings, speech and model list.
4. A **Cloudflare quick tunnel**, which gives the gateway a public `https://….trycloudflare.com` address.

It finishes by testing every job through the public address and printing the lines for FARSHORE's `.env`.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. Keep this tab open while you play: the server stops when the session ends, and the address and key change on every new run.

In [ ]:
#@title 1. Choose the models { display-mode: "form" }
PRESET = "Recommended: Qwen3.5 (T4)"  #@param ["Recommended: Qwen3.5 (T4)", "Gemma 4 planner (T4)", "Proven: Gemma 3 + Qwen3 (T4, measured)", "Fast: small models (any GPU)", "Max: Qwen3.5 27B (L4 or A100)", "Custom (the fields below)"]
#@markdown A preset sets the three chat models. **Custom** uses the fields below. Any Ollama model name works there.
PLANNER = "qwen3.5:9b"  #@param {type:"string"}
COMMENTARY = "qwen3.5:4b"  #@param {type:"string"}
ORACLE = "same as planner"  #@param {type:"string"}
#@markdown The Oracle's question writer can be a different model, at the cost of more GPU memory.
EMBEDDER = "embeddinggemma:300m"  #@param ["embeddinggemma:300m", "qwen3-embedding:0.6b", "nomic-embed-text", "off"]
#@markdown The embedding model ranks the *Odyssey* passages for the Oracle. `off` leaves the game on keyword search (BM25) alone.
ENABLE_VOICE = True  #@param {type:"boolean"}
#@markdown Leave the voice on for the bard and the rivals to speak with Kokoro. Without it the game uses the browser's built-in voices.
THINKING = False  #@param {type:"boolean"}
#@markdown Leave thinking **off**. A thinking model spends seconds and most of its token budget reasoning before it answers, and the game's requests have short time limits.
CONTEXT = 8192  #@param [4096, 8192, 16384] {type:"raw"}
#@markdown Context window per request. The game's longest prompts are well under 4,000 tokens; a smaller window leaves more GPU memory for the models.

PRESETS = {
    # name: (planner, commentary, oracle)
    "Recommended: Qwen3.5 (T4)": ("qwen3.5:9b", "qwen3.5:4b", "qwen3.5:9b"),
    "Gemma 4 planner (T4)": ("gemma4:12b", "qwen3.5:4b", "gemma4:12b"),
    "Proven: Gemma 3 + Qwen3 (T4, measured)": ("gemma3:12b", "qwen3:4b-instruct", "gemma3:12b"),
    "Fast: small models (any GPU)": ("qwen3.5:4b", "qwen3.5:2b", "qwen3.5:4b"),
    "Max: Qwen3.5 27B (L4 or A100)": ("qwen3.5:27b", "qwen3.5:4b", "qwen3.5:27b"),
}
if PRESET in PRESETS:
    PLANNER, COMMENTARY, ORACLE = PRESETS[PRESET]
PLANNER, COMMENTARY = PLANNER.strip(), COMMENTARY.strip()
ORACLE = PLANNER if ORACLE.strip() in ("", "same as planner") else ORACLE.strip()
EMBEDDER = EMBEDDER.strip()
CHAT_MODELS = list(dict.fromkeys([PLANNER, COMMENTARY, ORACLE]))
ALL_MODELS = CHAT_MODELS + ([EMBEDDER] if EMBEDDER != "off" else [])
print(f"Preset      {PRESET}")
for job, name in [("Planner", PLANNER), ("Commentary", COMMENTARY), ("Oracle", ORACLE), ("Embeddings", EMBEDDER), ("Voice", "Kokoro-82M" if ENABLE_VOICE else "off (browser voices)")]:
    print(f"{job:<11} {name}")

In [ ]:
#@title 2. Install Ollama, Kokoro, the gateway and the tunnel (2 to 4 minutes)
import subprocess, sys

def step(name, command, required=True):
    """Runs one install step. Colab hides a subprocess's own output, so a failure prints it here."""
    print(f"- {name}...", flush=True)
    result = subprocess.run(command, shell=isinstance(command, str), capture_output=True, text=True)
    if result.returncode == 0:
        return True
    print((result.stdout + result.stderr)[-3000:])
    if required:
        raise RuntimeError(f"{name} failed (exit code {result.returncode}). The lines above say why.")
    return False

step("System packages", "apt-get -qq update && apt-get -qq install -y zstd pciutils lshw espeak-ng")
# The latest Ollama: Qwen3.5, Gemma 4 and EmbeddingGemma all need a recent release.
step("Ollama", "curl -fsSL https://ollama.com/install.sh | sh")
print("  " + (subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip().splitlines() or ["version unknown"])[-1])
step("Gateway packages", [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "httpx", "soundfile", "numpy"])
if ENABLE_VOICE:
    # Kokoro supports Python 3.10 to 3.12. On a newer runtime the voice is skipped
    # and the game speaks with its own built-in Kokoro voice on your computer.
    ENABLE_VOICE = sys.version_info < (3, 13) and step("Kokoro voice", [sys.executable, "-m", "pip", "install", "-q", "kokoro>=0.9.4"], required=False)
    if not ENABLE_VOICE:
        print(f"  The Colab voice is off (Python {sys.version.split()[0]}). The game will use its built-in voice instead.")
step("Cloudflare tunnel", "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared")
print("Installed.")

In [ ]:
#@title 3. Check the GPU and whether the models fit
GPU = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
if GPU.returncode != 0:
    raise RuntimeError("No GPU. Choose Runtime → Change runtime type → T4 GPU, then run all cells again.")
GPU_NAME, VRAM_MB = [x.strip() for x in GPU.stdout.strip().splitlines()[0].split(",")]
VRAM_GB = int(VRAM_MB) / 1024
# Download sizes from ollama.com (GB), close to the memory each model's weights take on the GPU.
SIZES = {
    "qwen3.5:0.8b": 1.2, "qwen3.5:2b": 2.7, "qwen3.5:4b": 3.3, "qwen3.5:9b": 6.6, "qwen3.5:27b": 17, "qwen3.5:35b": 22,
    "gemma4:e2b": 4.6, "gemma4:e4b": 6.6, "gemma4:12b": 7.7, "gemma4:26b": 16, "gemma4:31b": 19,
    "gemma3:4b": 3.3, "gemma3:12b": 8.1, "qwen3:4b-instruct": 2.5, "qwen3:1.7b": 1.4, "qwen3:14b": 9.3, "gpt-oss:20b": 14,
    "embeddinggemma:300m": 0.6, "qwen3-embedding:0.6b": 0.6, "nomic-embed-text": 0.3,
}
# Each chat model also needs room for its context (the KV cache) and CUDA buffers.
need = sum(SIZES.get(m, 5.0) for m in ALL_MODELS) + 1.0 * len(CHAT_MODELS) + 0.5
print(f"{GPU_NAME}: {VRAM_GB:.0f} GB. These models need about {need:.1f} GB.")
for m in ALL_MODELS:
    print(f"  {m:<24} {SIZES.get(m, '?')} GB" + ("" if m in SIZES else "  (size unknown; assumed 5 GB)"))
if need > VRAM_GB * 0.95:
    print(f"\n⚠ That is more than this GPU holds. Ollama will swap models in and out or run part of one on the CPU, which makes the game wait."
          f"\n  Choose a smaller preset in step 1 (\"Fast\" on a T4), or a bigger GPU (\"Max\" wants an L4 or A100).")
else:
    print("They fit.")

In [ ]:
#@title 4. Start Ollama, download the models and load them onto the GPU
import json, os, time, urllib.request

def local(path, payload=None, timeout=600):
    request = urllib.request.Request("http://127.0.0.1:11434" + path, data=None if payload is None else json.dumps(payload).encode(),
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read() or b"{}")

ollama_log = open("/content/ollama.log", "w")
ollama = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT, env={
    **os.environ,
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_KEEP_ALIVE": "24h",              # stay loaded while you play
    "OLLAMA_NUM_PARALLEL": "2",              # commentary can arrive while the planner works
    "OLLAMA_MAX_LOADED_MODELS": str(len(ALL_MODELS) + 1),
    "OLLAMA_CONTEXT_LENGTH": str(CONTEXT),
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_KV_CACHE_TYPE": "q8_0",          # half the context memory of f16, with little loss
})
for _ in range(60):
    try:
        print("Ollama", local("/api/version", timeout=2)["version"])
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start. See /content/ollama.log")

for name in ALL_MODELS:
    print(f"- Downloading {name}...", flush=True)
    pull = subprocess.run(["ollama", "pull", name], capture_output=True, text=True)
    if pull.returncode != 0:
        print(pull.stderr[-1500:])
        raise RuntimeError(f"Could not download {name}. Check the name on https://ollama.com/library, or choose another preset in step 1.")
for name in ALL_MODELS:
    # Load each model now, so the first rival, line and question are not slow.
    started = time.time()
    if name == EMBEDDER:
        local("/api/embed", {"model": name, "input": "ready", "keep_alive": "24h"})
    else:
        local("/api/generate", {"model": name, "keep_alive": "24h", "stream": False})
    print(f"  {name} loaded in {time.time() - started:.0f} s")

tag = lambda m: m if ":" in m else m + ":latest"

def placement():
    """Each loaded model's share on the GPU: anything under 100% runs partly on the CPU."""
    rows = []
    for m in local("/api/ps").get("models", []):
        rows.append((m["name"], m.get("size", 0) / 1e9, 100 * m.get("size_vram", 0) / max(1, m.get("size", 1))))
    return rows

print()
for name, size, on_gpu in placement():
    print(f"  {name:<24} {size:5.1f} GB  {on_gpu:3.0f}% on the GPU" + ("" if on_gpu > 99 else "  ⚠ partly on the CPU: slower"))
missing = [m for m in ALL_MODELS if tag(m) not in [tag(p[0]) for p in placement()]]
if missing:
    print(f"  ⚠ Not resident: {', '.join(missing)}. The GPU could not hold every model at once; Ollama will reload them in turn.")

In [ ]:
#@title 5. Start the gateway (API key, chat, embeddings and voice)
import secrets, sys
GATEWAY_KEY = secrets.token_urlsafe(24)
open("/content/gateway.py", "w").write(r"""
import io, json, os, secrets, time
import httpx, numpy as np, soundfile as sf
from fastapi import FastAPI, Header, HTTPException, Request, Response
from pydantic import BaseModel

KEY = os.environ["FARSHORE_GATEWAY_KEY"]
VOICE = os.environ.get("FARSHORE_VOICE") == "1"
THINKING = os.environ.get("FARSHORE_THINKING") == "1"
LOG = "/content/requests.log"
app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
ollama = httpx.AsyncClient(base_url="http://127.0.0.1:11434", timeout=httpx.Timeout(300, connect=5))
pipelines = {}

def check(authorization):
    if not authorization or not secrets.compare_digest(authorization.encode(), ("Bearer " + KEY).encode()):
        raise HTTPException(status_code=401, detail="Missing or wrong API key.")

def note(path, model, status, started, detail=""):
    with open(LOG, "a") as f:
        f.write(json.dumps({"time": time.strftime("%H:%M:%S"), "path": path, "model": model, "status": status, "ms": round((time.time() - started) * 1000), "detail": detail}) + "\n")

async def body_of(request, limit):
    raw = await request.body()
    if len(raw) > limit:
        raise HTTPException(status_code=413, detail="Request too large.")
    try:
        return json.loads(raw)
    except ValueError:
        raise HTTPException(status_code=400, detail="The body must be JSON.")

@app.get("/health")
async def health():
    # No key needed, and nothing revealed: just whether the server is up.
    try:
        await ollama.get("/api/version", timeout=5)
        return {"ok": True}
    except Exception:
        return Response('{"ok": false}', 503, media_type="application/json")

@app.get("/v1/models")
async def models(authorization: str = Header(None)):
    check(authorization)
    r = await ollama.get("/v1/models")
    return Response(r.content, r.status_code, media_type="application/json")

@app.post("/v1/chat/completions")
async def chat(request: Request, authorization: str = Header(None)):
    check(authorization)
    started, body = time.time(), await body_of(request, 64_000)
    if body.get("stream"):
        raise HTTPException(status_code=400, detail="This gateway answers whole replies only (stream: false).")
    # Thinking off unless the caller asks: the game's requests have short time limits.
    if not THINKING and "reasoning_effort" not in body:
        body["reasoning_effort"] = "none"
    r = await ollama.post("/v1/chat/completions", json=body)
    if r.status_code == 400 and "think" in r.text.lower() and body.get("reasoning_effort") == "none":
        body.pop("reasoning_effort")  # a model without a thinking switch
        r = await ollama.post("/v1/chat/completions", json=body)
    tokens = ""
    try:
        usage = r.json().get("usage", {})
        tokens = f"{usage.get('prompt_tokens', '?')} in, {usage.get('completion_tokens', '?')} out"
    except Exception:
        pass
    note("chat", body.get("model"), r.status_code, started, tokens)
    return Response(r.content, r.status_code, media_type="application/json")

@app.post("/v1/embeddings")
async def embeddings(request: Request, authorization: str = Header(None)):
    check(authorization)
    started, body = time.time(), await body_of(request, 1_000_000)
    items = body.get("input")
    count = len(items) if isinstance(items, list) else 1
    if count > 256:
        raise HTTPException(status_code=413, detail="At most 256 texts per request.")
    r = await ollama.post("/v1/embeddings", json=body)
    note("embeddings", body.get("model"), r.status_code, started, f"{count} texts")
    return Response(r.content, r.status_code, media_type="application/json")

class Speech(BaseModel):
    input: str
    voice: str = "bm_george"
    speed: float = 1.0
    model: str | None = None
    response_format: str | None = None

@app.post("/v1/audio/speech")
def speech(body: Speech, authorization: str = Header(None)):
    check(authorization)
    if not VOICE:
        raise HTTPException(status_code=503, detail="Voice is turned off in this notebook.")
    started, text = time.time(), body.input.strip()[:400]
    if not text:
        raise HTTPException(status_code=400, detail="Empty input.")
    from kokoro import KPipeline
    lang = "b" if body.voice.startswith("b") else "a"
    if lang not in pipelines:
        pipelines[lang] = KPipeline(lang_code=lang)
    parts = []
    for _, _, audio in pipelines[lang](text, voice=body.voice, speed=max(0.5, min(2.0, body.speed))):
        parts.append(audio.detach().cpu().numpy() if hasattr(audio, "detach") else np.asarray(audio))
    if not parts:
        raise HTTPException(status_code=500, detail="No audio was produced.")
    buffer = io.BytesIO()
    sf.write(buffer, np.concatenate(parts), 24000, format="WAV")
    note("speech", body.voice, 200, started, f"{len(text)} characters")
    return Response(buffer.getvalue(), media_type="audio/wav")
""")
gateway_log = open("/content/gateway.log", "w")
gateway = subprocess.Popen([sys.executable, "-m", "uvicorn", "gateway:app", "--host", "127.0.0.1", "--port", "8000"],
                           cwd="/content", stdout=gateway_log, stderr=subprocess.STDOUT,
                           env={**os.environ, "FARSHORE_GATEWAY_KEY": GATEWAY_KEY, "FARSHORE_VOICE": "1" if ENABLE_VOICE else "0", "FARSHORE_THINKING": "1" if THINKING else "0"})
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("The gateway did not start. See /content/gateway.log")
print("Gateway running on port 8000.")

In [ ]:
#@title 6. Open the public tunnel
import re
tunnel_log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=tunnel_log, stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(60):
    time.sleep(1)
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if found:
        PUBLIC_URL = found.group(0)
        break
if not PUBLIC_URL:
    raise RuntimeError("No tunnel address yet. See /content/tunnel.log and run this cell again.")
# A new address can take a few seconds to resolve everywhere.
for _ in range(30):
    try:
        urllib.request.urlopen(PUBLIC_URL + "/health", timeout=5)
        break
    except Exception:
        time.sleep(2)
print("Public address:", PUBLIC_URL)

In [ ]:
#@title 7. Test every job through the public address, then copy the lines into FARSHORE's .env
import math, statistics

def call(path, payload=None, timeout=180):
    request = urllib.request.Request(PUBLIC_URL + path, data=None if payload is None else json.dumps(payload).encode(),
                                     headers={"Authorization": "Bearer " + GATEWAY_KEY, "Content-Type": "application/json", "User-Agent": "farshore-colab"})
    started = time.time()
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read(), time.time() - started

def ask(model, schema, system, user, max_tokens, name="farshore_test"):
    """The same request shape the game sends (server/lib/chat.ts): JSON against a strict schema."""
    raw, seconds = call("/v1/chat/completions", {
        "model": model, "stream": False, "temperature": 0.7, "max_tokens": max_tokens,
        "response_format": {"type": "json_schema", "json_schema": {"name": name, "strict": True, "schema": schema}},
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]})
    reply = json.loads(raw)
    message = reply["choices"][0]["message"]
    content = re.sub(r"<think>[\s\S]*?</think>", "", message.get("content") or "").strip()
    thought = bool(message.get("reasoning") or message.get("reasoning_content") or "<think>" in (message.get("content") or ""))
    return json.loads(content), seconds, reply.get("usage", {}).get("completion_tokens"), thought

problems = []
def result(job, ok, text):
    print(("✓ " if ok else "✗ ") + f"{job}: {text}")
    if not ok:
        problems.append(job)

served = [d["id"] for d in json.loads(call("/v1/models")[0])["data"]]
print("Served models:", ", ".join(served))
for m in ALL_MODELS:
    if tag(m) not in map(tag, served):
        result(m, False, "not served: run step 4 again")
print()

# Planner: a rival decision from a fixed menu, as in Ithaca.
menu = {"type": "object", "properties": {"optionId": {"type": "string", "enum": ["high-left", "steady-center", "low-right"]}, "reason": {"type": "string", "maxLength": 160}}, "required": ["optionId", "reason"], "additionalProperties": False}
try:
    r, s, n, thought = ask(PLANNER, menu, "You are Antinous in an archery contest. Choose one option from the menu and give one short reason that uses only the facts given.",
                           "Wind: 6 knots from the left. Your last arrow landed 2 rings low. Options: high-left, steady-center, low-right.", 120, "farshore_decision")
    result(f"Planner ({PLANNER})", r["optionId"] in menu["properties"]["optionId"]["enum"], f"{s:.1f} s, {n} tokens{', thought first' if thought else ''}: {r['optionId']} — {r['reason']}")
except Exception as e:
    result(f"Planner ({PLANNER})", False, f"{type(e).__name__}: {e}")

# Commentary: one short live line.
line = {"type": "object", "properties": {"line": {"type": "string", "maxLength": 140}}, "required": ["line"], "additionalProperties": False}
try:
    r, s, n, thought = ask(COMMENTARY, line, "You are the bard in FARSHORE, an arcade game based on the Odyssey. Say one short, vivid line about what just happened.",
                           "Odysseus slipped past the blind Cyclops under a ram.", 70, "farshore_commentary")
    result(f"Commentary ({COMMENTARY})", 0 < len(r["line"]) <= 140, f"{s:.1f} s{', thought first' if thought else ''}: {r['line']}")
except Exception as e:
    result(f"Commentary ({COMMENTARY})", False, f"{type(e).__name__}: {e}")

# The Oracle: one question from two passages, checked the way the game checks it (server/oracle/oracle.ts).
P1 = ("Therefore pass these Sirens by, and stop your men’s ears with wax that none of them may hear; but if you like you can listen yourself, "
      "for you may get the men to bind you as you stand upright on a cross piece half way up the mast, and they must lash the rope’s ends to the mast itself, "
      "that you may have the pleasure of listening. If you beg and pray the men to unloose you, then they must bind you faster.")
P2 = ("“I had hardly finished telling everything to the men before we reached the island of the two Sirens, for the wind had been very favourable. "
      "Then all of a sudden it fell dead calm; there was not a breath of wind nor a ripple upon the water, so the men furled the sails and stowed them; "
      "then taking to their oars they whitened the water with the foam they raised in rowing. Meanwhile I look a large wheel of wax and cut it up small with my sword.")
oracle_schema = {"type": "object", "properties": {
    "question": {"type": "string", "maxLength": 200}, "options": {"type": "array", "items": {"type": "string", "maxLength": 90}, "minItems": 4, "maxItems": 4},
    "answer": {"type": "integer", "enum": [0, 1, 2, 3]}, "source": {"type": "string", "enum": ["P1", "P2"]},
    "evidence": {"type": "string", "maxLength": 300}, "explanation": {"type": "string", "maxLength": 240}},
    "required": ["question", "options", "answer", "source", "evidence", "explanation"], "additionalProperties": False}
oracle_system = ("You are the Oracle in FARSHORE, an arcade game based on Homer's Odyssey. Before Odysseus may enter \"The Sirens\", you test him with one multiple-choice question.\n"
                 "Write the question from the passages given, and from nothing else. It must be answerable from a single passage.\n"
                 "Give exactly four short options: one correct, and three plausible but clearly wrong according to that passage.\n"
                 "\"answer\" is the index (0 to 3) of the correct option. \"source\" names the passage it comes from (P1 or P2).\n"
                 "\"evidence\" must be copied word for word from that passage: one phrase or sentence, at most 30 words, that shows the answer is right.\n"
                 "\"explanation\" is one sentence for the player, after they answer.\n"
                 "Use the names as the passages write them. Do not ask about passage numbers, books or translators. Do not put the answer in the question.")
normalize = lambda s: re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]+", " ", re.sub(r"[“”\"]", " ", re.sub(r"[‘’`']", "", s.lower())))).strip()
try:
    tries = []
    for attempt in range(3):
        r, s, n, thought = ask(ORACLE, oracle_schema, oracle_system, f"Encounter: The Sirens (the Sirens)\n\nP1 (Book XII, passage 5): {P1}\n\nP2 (Book XII, passage 15): {P2}", 320, "farshore_oracle")
        cited = P1 if r["source"] == "P1" else P2
        words = len(normalize(r["evidence"]).split())
        found = normalize(r["evidence"]) in normalize(cited)
        quoted = found and 4 <= words <= 40
        distinct = len({normalize(o) for o in r["options"]}) == 4
        tries.append(s)
        if quoted and distinct and r["question"].strip().endswith("?"):
            break
    result(f"Oracle ({ORACLE})", quoted and distinct, f"{len(tries)} attempt(s), {statistics.mean(tries):.1f} s each{', thought first' if thought else ''}")
    print(f"    Q: {r['question']}")
    for i, o in enumerate(r["options"]):
        print(f"    {'ABCD'[i]}. {o}{'   ← answer' if i == r['answer'] else ''}")
    verdict = "found word for word" if quoted else f"too long: {words} words, the limit is 40" if found or words > 40 else "not in the passage as written"
    print(f"    Evidence ({r['source']}, {verdict}): “{r['evidence']}”")
    if not quoted:
        print("    The game sets a question like this aside and asks again. A model that often fails here leaves the Oracle slow or silent.")
except Exception as e:
    result(f"Oracle ({ORACLE})", False, f"{type(e).__name__}: {e}")

# The Oracle's passage search: the answering passage must score higher, with the prompts the game uses (server/oracle/retrieval.ts).
if EMBEDDER != "off":
    def prompt(kind, text):
        if "embeddinggemma" in EMBEDDER:
            return f"task: search result | query: {text}" if kind == "query" else f"title: none | text: {text}"
        if "qwen3-embedding" in EMBEDDER:
            return f"Instruct: Given a question about Homer's Odyssey, retrieve the passage that answers it\nQuery: {text}" if kind == "query" else text
        if "nomic-embed" in EMBEDDER:
            return f"search_query: {text}" if kind == "query" else f"search_document: {text}"
        return text
    try:
        lotus = "On the tenth day we reached the land of the Lotus-eaters, who live on a food that comes from a kind of flower."
        raw, s = call("/v1/embeddings", {"model": EMBEDDER, "input": [prompt("query", "How did the crew avoid hearing the deadly singing?"), prompt("document", P1), prompt("document", lotus)]})
        v = [d["embedding"] for d in sorted(json.loads(raw)["data"], key=lambda d: d.get("index", 0))]
        cos = lambda a, b: sum(x * y for x, y in zip(a, b)) / (math.hypot(*a) * math.hypot(*b))
        right, wrong = cos(v[0], v[1]), cos(v[0], v[2])
        result(f"Passage search ({EMBEDDER})", right > wrong, f"{s:.2f} s, {len(v[0])} dimensions; the Sirens passage {right:.2f} against the Lotus passage {wrong:.2f}")
    except Exception as e:
        result(f"Passage search ({EMBEDDER})", False, f"{type(e).__name__}: {e}")

if ENABLE_VOICE:
    try:
        audio, s = call("/v1/audio/speech", {"input": "Sing to me, Muse, of the man of many ways.", "voice": "bm_george"})
        result("Voice (Kokoro)", audio[:4] == b"RIFF", f"{s:.1f} s, {len(audio) // 1024} KB of audio")
    except Exception as e:
        result("Voice (Kokoro)", False, f"{type(e).__name__}: {e}")

print()
if problems:
    print("⚠ Not everything passed:", "; ".join(problems))
    print("  A model that fails here will fail in the game too. Try another preset in step 1, or see Troubleshooting below.\n")
print("# ---- paste into FARSHORE/.env (replace any old lines), then restart the game server ----")
print(f"FARSHORE_MODEL_URL={PUBLIC_URL}/v1/chat/completions")
print(f"FARSHORE_MODEL_NAME={PLANNER}")
if COMMENTARY != PLANNER:
    print(f"FARSHORE_COMMENTARY_MODEL={COMMENTARY}")
if ORACLE != PLANNER:
    print(f"FARSHORE_ORACLE_MODEL={ORACLE}")
if EMBEDDER == "off":
    print("FARSHORE_EMBED_URL=off")
else:
    print(f"# The Oracle searches with {PUBLIC_URL}/v1/embeddings and the same key.")
    print(f"FARSHORE_EMBED_MODEL={EMBEDDER}")
print(f"FARSHORE_API_KEY={GATEWAY_KEY}")
if ENABLE_VOICE:
    print(f"FARSHORE_TTS_URL={PUBLIC_URL}/v1/audio/speech")

In [ ]:
#@title 8. (Any time) Is it still running? What has the game asked?
try:
    urllib.request.urlopen(PUBLIC_URL + "/health", timeout=10)
    print("Tunnel and gateway: up at", PUBLIC_URL)
except Exception as e:
    print("⚠ The public address does not answer:", e, "\n  If the session restarted, run all cells again and update .env.")
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu", "--format=csv"], capture_output=True, text=True).stdout)
for name, size, on_gpu in placement():
    print(f"  {name:<24} {size:5.1f} GB  {on_gpu:3.0f}% on the GPU")
print("\nLast requests (time, job, model, status, milliseconds):")
try:
    for row in [json.loads(l) for l in open("/content/requests.log").read().splitlines()[-12:]]:
        print(f"  {row['time']}  {row['path']:<10} {str(row['model']):<22} {row['status']}  {row['ms']:>6} ms  {row['detail']}")
except FileNotFoundError:
    print("  None yet. Start an encounter in FARSHORE.")

In [ ]:
#@title 9. (Optional) Measure each job's speed on this GPU
RUNS = 5  #@param {type:"integer"}
def timed(job, fn):
    seconds = []
    for _ in range(RUNS):
        started = time.time()
        fn()
        seconds.append(time.time() - started)
    seconds.sort()
    print(f"{job:<34} median {statistics.median(seconds):5.2f} s   slowest {seconds[-1]:5.2f} s")
timed(f"Planner decision ({PLANNER})", lambda: ask(PLANNER, menu, "You are Antinous. Choose one option and give one short reason.", "Wind 6 knots from the left. Options: high-left, steady-center, low-right.", 120))
timed(f"Commentary line ({COMMENTARY})", lambda: ask(COMMENTARY, line, "You are the bard. One short line.", "Odysseus strung the bow.", 70))
timed(f"Oracle question ({ORACLE})", lambda: ask(ORACLE, oracle_schema, oracle_system, f"Encounter: The Sirens\n\nP1 (Book XII, passage 5): {P1}\n\nP2 (Book XII, passage 15): {P2}", 320))
if EMBEDDER != "off":
    timed(f"Embed 32 passages ({EMBEDDER})", lambda: call("/v1/embeddings", {"model": EMBEDDER, "input": [prompt("document", P1)] * 32}))
if ENABLE_VOICE:
    timed("Voice line (Kokoro)", lambda: call("/v1/audio/speech", {"input": "The gate opens.", "voice": "bm_george"}))

### Notes

- **Keep this tab open.** The server stops when the Colab session ends or disconnects (free sessions also stop after a while idle). For a new session, run all cells again: the address and key change each time, so update `.env` again and **restart the game server**, because it reads `.env` only when it starts.
- **The key matters.** The tunnel address is public, but the gateway rejects any request without your key, and the key is shown only in this notebook. Don't share the printed lines.
- **Thinking is off.** The gateway asks every model to answer directly (`reasoning_effort: "none"`), unless you tick `THINKING` in step 1 or the game asks otherwise. Thinking models answer better on hard puzzles, but they take seconds longer, and a short reply budget such as the commentary's 70 tokens can run out before they answer.
- **The Oracle's Key** uses two endpoints from here: `/v1/chat/completions` writes the question, and `/v1/embeddings` ranks the passages. The game derives the embeddings address from `FARSHORE_MODEL_URL`. The first question after the game server starts also builds the passage index, about 2 seconds on this GPU (about a minute on a laptop CPU). It is saved under `.cache/oracle`, so this happens once per embedding model.
- **Online accounts.** On the hosted FARSHORE site, players add their own connection under Account. Their model then writes the Oracle's questions too, while the passage search stays on the server's own embedding service.

### Choosing models by measurement

With the game's repository on your computer, after pasting the `.env` lines:

```
node --experimental-strip-types scripts/ai-eval.mjs --url <FARSHORE_MODEL_URL> --model <planner> --key <FARSHORE_API_KEY>
node --experimental-strip-types scripts/rag-eval.mjs --embed <PUBLIC_URL>/v1/embeddings --model embeddinggemma:300m,qwen3-embedding:0.6b --key <FARSHORE_API_KEY>
```

- **`ai-eval.mjs`** sends about 100 of the game's real requests through its own checks: rival decisions, debriefs, Athena (including injection attempts) and 16 Oracle questions. It reports how many were valid, how many reasons were grounded in the game's numbers, how many Oracle questions passed every check on the first try, and the latency.
- **`rag-eval.mjs`** measures the passage search with hit@1, hit@3 and MRR on 25 labelled questions.

Results are saved under `docs/ai-eval/`. To compare chat models, pull another into this server (`!ollama pull <name>` in a new cell) and pass its name as `--model`.

### Troubleshooting

| You see | It means | Do this |
|---|---|---|
| `412: requires a newer version of Ollama` | The install step got an old Ollama | Run step 2 again. Colab always installs the latest. |
| "partly on the CPU" in step 4 | The models do not all fit on the GPU | Choose a smaller preset (**Fast**), or set `ORACLE` to the planner |
| The game says the model cannot be reached; the address gives a DNS error | The Colab session ended | Run all cells again and update `.env` |
| The address in a browser says `Method Not Allowed` or `{"detail":"Not Found"}` | The gateway is up; browsers send GET | Nothing to fix: the game sends POST with the key |
| `401` | `.env` has an old key | Copy the new lines from step 7 and restart the game server |
| The Oracle says "keywords only" | The embedding service did not answer | Check `FARSHORE_EMBED_MODEL` matches step 1, then restart the game server |